# DSA211 — Predicting Online Purchases
### Topic 2 · Classification System Evaluation · Revised analysis

**Question.** Can browsing-session data help an online retailer identify visitors likely to purchase, so that live sales support can be prioritised?

**Response:** `Revenue` = 1 if a session ends in a purchase, otherwise 0. **Unit of analysis:** browsing session, not a uniquely identified customer.

**Scope:** logistic regression, Naive Bayes, LDA and a QDA comparison; EDA; initial all-predictor models; cross-validation-guided improvement; ROC/AUC, confusion matrices, sensitivity/specificity; model selection; **(Extension 1)** a cost-justified threshold; **(Extension 2)** CV versus held-out evaluation. No KNN, gradient boosting, splines, bootstrap thresholds or learning curves in this version, as agreed for Phase 2.

**Source and protocol:** the supplied two-page *Predicting Online Purchases* project brief, with the professor's subsequent instruction to first attempt *all predictors* in *all models*. The approved protocol keeps the historical **75/25 stratified split (seed 42)** and uses **5-fold stratified CV** on the training data. The original notebook is preserved unchanged.

> **Historical holdout warning.** The older notebook already displayed results for this same 25% test partition. This analysis uses no test observations for fitting, CV, feature decisions or threshold choice, but the final test score is a **retrospective check**, not a new, previously unseen confirmatory evaluation. That limitation must appear in any submitted write-up.

## 0. Setup and pre-declared decision rules

**Objective:** fix the evaluation rules *before* inspecting model comparisons.

- **Primary discrimination metric:** CV ROC-AUC (ranking buyers above non-buyers).
- **Operational metrics:** sensitivity, specificity, confusion matrix and expected classification-error cost.
- **Cost assumption from the brief:** `C_FN / C_FP = 5`, normalising a false positive to cost 1 and a false negative to cost 5. This is a scenario assumption, **not** a measured CAC or AOV estimate.
- **CV:** 5 stratified folds, each used once as validation; imputation, encoding, transformation and parameter selection fit only in the corresponding training fold.
- **Selection:** compare CV AUC for all eligible configurations; show error cost as an additional explicit consideration. The best AUC specification is the default predictive winner. Choose its operating threshold on **training out-of-fold** probabilities, not test observations.

We deliberately do not assume that the previous notebook's chosen LDA model or threshold will be selected again.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, PowerTransformer, QuantileTransformer, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, brier_score_loss

SEED, TEST_FRAC, K = 42, .25, 5
C_FP, C_FN = 1.0, 5.0
FOLDS = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
NUM_ALL = ['Administrative','Administrative_Duration','Informational','Informational_Duration',
           'ProductRelated','ProductRelated_Duration','BounceRates','ExitRates','PageValues']
CAT_ALL = ['SpecialDay','Month','OperatingSystems','Browser','Region','TrafficType','VisitorType','Weekend']
ALL = NUM_ALL + CAT_ALL
PROBLEM_COL = 'PageValues'
NUM_SAFE = [c for c in NUM_ALL if c != PROBLEM_COL]
SAFE = NUM_SAFE + CAT_ALL
pd.set_option('display.max_columns', 24)
print(f'Primary metric: CV ROC-AUC | Stratified {K}-fold CV | test fraction={TEST_FRAC:.0%} | seed={SEED}')
print(f'Brief cost scenario: C_FP={C_FP:g}, C_FN={C_FN:g}, implied theoretical threshold={C_FP/(C_FN+C_FP):.3f} if calibrated')

## 1. Load, check quality, and split once

**Why first?** The professor's CV workflow permits dataset-quality checks before splitting, but predictor-outcome EDA and modelling decisions belong to the training partition. We do **not** drop duplicate observations by default: identical feature rows do not prove duplicate customer sessions.

**Python:** `train_test_split(..., stratify=y)` preserves the approximate purchase rate across partitions. `random_state=42` reproduces the historical 75/25 partition.

In [ ]:
path = Path('data/uci_468.csv')
if not path.is_file():
    raise FileNotFoundError('Run this notebook from the repository root; expected data/uci_468.csv')
raw = pd.read_csv(path)
assert set(ALL + ['Revenue']) == set(raw.columns), 'Unexpected dataset columns — inspect source before proceeding.'
assert len(raw) == 12330, 'Dataset size differs from the project brief — inspect before proceeding.'
quality = pd.DataFrame({'dtype': raw.dtypes.astype(str), 'missing': raw.isna().sum(),
                        'unique': raw.nunique(dropna=True)})
print('Shape:', raw.shape, ' | duplicate full records:', int(raw.duplicated().sum()))
print('Outcome:', raw['Revenue'].value_counts(dropna=False).to_dict())
display(quality)
X = raw[ALL].copy()
# IDs are nominal, not quantities; preserve values while marking categorical semantics.
for name in CAT_ALL:
    X[name] = X[name].astype(str)
y = raw['Revenue'].astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_FRAC,
                                     stratify=y, random_state=SEED)
Xtr, Xte, ytr, yte = (z.reset_index(drop=True) for z in (Xtr, Xte, ytr, yte))
del raw, X, y
assert len(Xtr) == 9247 and len(Xte) == 3083
print(f'Training: {len(Xtr):,} sessions, {ytr.sum():,} purchases ({ytr.mean():.2%})')
print(f'Holdout: {len(Xte):,} sessions — reserved until Extension 2')

## 2. Exploratory data analysis — training only

**Objective:** identify patterns relevant to classification and propose *testable*, rather than automatic, modifications.

**Questions:** Is the outcome imbalanced? Which numerical variables contain many zeros or heavy right tails? Does purchase rate vary across categories? Are technical identifiers being interpreted correctly?

In [ ]:
no_buy, buy = ytr.value_counts().reindex([0, 1]).astype(int).values
fig, ax = plt.subplots(figsize=(5.8, 3.0))
ax.bar(['No purchase', 'Purchase'], [no_buy, buy])
ax.set(title='Outcome balance, training data', ylabel='Sessions')
plt.show()
print(f'Majority-class accuracy: {no_buy/len(ytr):.2%}; its sensitivity is exactly zero.')
num_summary = pd.DataFrame({
    'median': Xtr[NUM_ALL].median(),
    'zero %': (Xtr[NUM_ALL] == 0).mean().mul(100),
    'skewness': Xtr[NUM_ALL].skew(),
    'median (buyers)': Xtr.loc[ytr.eq(1), NUM_ALL].median(),
    'median (non-buyers)': Xtr.loc[ytr.eq(0), NUM_ALL].median(),
}).round(3)
display(num_summary)
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for ax, name in zip(axes.flat, NUM_SAFE):
    values = np.log1p(Xtr[name].clip(lower=0))
    ax.hist(values[ytr == 0], bins=25, alpha=.55, density=True, label='no purchase')
    ax.hist(values[ytr == 1], bins=25, alpha=.45, density=True, label='purchase')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('log(1 + value)', fontsize=8)
axes.flat[0].legend(fontsize=8)
plt.suptitle('Eight behaviour measures by outcome (training only)')
plt.tight_layout(); plt.show()

In [ ]:
cat_summary = []
for name in CAT_ALL:
    rates = pd.DataFrame({'level': Xtr[name], 'purchase': ytr}).groupby('level').agg(
        sessions=('purchase','size'), purchase_rate=('purchase','mean')).sort_values('sessions', ascending=False)
    cat_summary.append({'feature': name, 'levels': rates.shape[0], 'largest category %':
                        100 * rates['sessions'].max() / len(Xtr),
                        'min purchase rate %': 100*rates.purchase_rate.min(),
                        'max purchase rate %': 100*rates.purchase_rate.max()})
display(pd.DataFrame(cat_summary).set_index('feature').round(2))
fig, axs = plt.subplots(1, 2, figsize=(12, 3.4))
for ax, name in zip(axs, ['Month', 'VisitorType']):
    p = pd.DataFrame({'category':Xtr[name], 'purchase':ytr}).groupby('category').purchase.agg(['mean','size'])
    ax.bar(p.index, p['mean'])
    ax.set(title=f'Purchase rate by {name}', ylabel='Purchase fraction')
    ax.tick_params(axis='x', rotation=40)
plt.tight_layout(); plt.show()
print('EDA implications: use stratification because purchases are uncommon; test transformations for skew/zeros;')
print('one-hot encode nominal categories inside each CV fold; examine PageValues before using it operationally.')

## 3. Modelling machinery and metrics

**Why a Pipeline?** If an encoder or scaler is fit before CV, validation rows influence its fitted parameters. Here, each fold refits its own preprocessing on the four training folds only.

- **Technical representation**: nominal categories are one-hot encoded (required to use numeric-input estimators); numeric columns are imputed/scaled inside CV.
- **First-pass meaning of "as-is"**: no predictor dropped, no skewness transformations, default model specifications. Encoding is not a predictive-improvement experiment.
- **AUC** summarises ranking over all thresholds. At the explicit threshold, confusion matrices provide true/false positives and negatives, sensitivity and specificity.

In [ ]:
class LogAndZeroIndicators(BaseEstimator, TransformerMixin):
    """Within each training fold: log1p numeric features and append >0 flags if >=50% zero."""
    def fit(self, X, y=None):
        A = np.asarray(X, float)
        self.zero_columns_ = np.flatnonzero((A == 0).mean(axis=0) >= .50)
        return self
    def transform(self, X):
        A = np.asarray(X, float)
        return np.column_stack([np.log1p(np.maximum(A, 0)), (A[:, self.zero_columns_] > 0).astype(float)])

class ColumnSubset(BaseEstimator, TransformerMixin):
    def __init__(self, columns): self.columns = columns
    def fit(self, X, y=None): return self
    def transform(self, X): return X.loc[:, self.columns].copy()

def build(model, columns, numeric_transform='raw', poly=False):
    nums = [c for c in NUM_ALL if c in columns]
    cats = [c for c in CAT_ALL if c in columns]
    numeric = [('impute', SimpleImputer(strategy='median'))]
    if numeric_transform == 'log1p':
        numeric.append(('transform', FunctionTransformer(np.log1p, validate=False)))
    elif numeric_transform == 'yeojohnson':
        numeric.append(('transform', PowerTransformer(method='yeo-johnson', standardize=False)))
    elif numeric_transform == 'quantile':
        numeric.append(('transform', QuantileTransformer(n_quantiles=200, output_distribution='normal', random_state=SEED)))
    elif numeric_transform == 'zeroind':
        numeric.append(('transform', LogAndZeroIndicators()))
    elif numeric_transform != 'raw':
        raise ValueError(f'Unknown transform: {numeric_transform}')
    if poly:
        numeric.append(('interactions', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)))
    numeric.append(('scale', StandardScaler()))
    pre = ColumnTransformer([
        ('numeric', Pipeline(numeric), nums),
        ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                                  ('encode', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))]), cats)
    ], remainder='drop', sparse_threshold=0.0)
    return Pipeline([('subset', ColumnSubset(list(columns))), ('pre', pre), ('clf', clone(model))])

def confusion_rates(y_true, prob, t=.5):
    tn, fp, fn, tp = confusion_matrix(y_true, np.asarray(prob) >= t, labels=[0,1]).ravel()
    return {'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
            'sensitivity': tp/(tp+fn) if tp+fn else np.nan,
            'specificity': tn/(tn+fp) if tn+fp else np.nan}

def cost_at(y_true, prob, t, fn_cost=C_FN, fp_cost=C_FP):
    r = confusion_rates(y_true, prob, t)
    return (fn_cost*r['FN'] + fp_cost*r['FP']) / len(y_true)

def optimal_threshold(y_true, prob, ratio=C_FN/C_FP):
    grid = np.unique(np.r_[0.0, np.linspace(.01, .99, 99), 1.0])
    costs = [cost_at(y_true, prob, t, ratio, 1.0) for t in grid]
    return float(grid[int(np.argmin(costs))])

MODELS = {
    'Logistic': LogisticRegression(max_iter=3000),
    'Gaussian NB': GaussianNB(),
    'LDA': LinearDiscriminantAnalysis(),
    'QDA': QuadraticDiscriminantAnalysis(),
}
LEDGER, CONFIGS, OOF_CACHE = [], {}, {}

def evaluate(label, model, columns, transform='raw', poly=False, stage='baseline'):
    if label in CONFIGS: raise ValueError('Duplicate experiment label: ' + label)
    p = build(model, columns, numeric_transform=transform, poly=poly)
    CONFIGS[label] = p
    try:
        with warnings.catch_warnings(record=True) as flagged:
            warnings.simplefilter('always')
            fold_auc = cross_val_score(p, Xtr, ytr, cv=FOLDS, scoring='roc_auc', n_jobs=1, error_score='raise')
        if not np.all(np.isfinite(fold_auc)): raise ValueError('Non-finite AUC')
        row = {'stage':stage, 'candidate':label, 'CV AUC':float(fold_auc.mean()),
               'CV SD':float(fold_auc.std(ddof=1)), 'status':'OK',
               'warnings':len(flagged), 'detail':''}
    except Exception as e:
        row = {'stage':stage, 'candidate':label, 'CV AUC':np.nan, 'CV SD':np.nan,
               'status':'FAILED', 'warnings':0, 'detail': f'{type(e).__name__}: {str(e).splitlines()[0][:120]}'}
    LEDGER.append(row)
    return row

def oof(label):
    if label not in OOF_CACHE:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            v = cross_val_predict(CONFIGS[label], Xtr, ytr, cv=FOLDS,
                                  method='predict_proba', n_jobs=1)[:, 1]
        if not np.all(np.isfinite(v)): raise ValueError('Non-finite predictions: '+label)
        OOF_CACHE[label] = v
    return OOF_CACHE[label]

print('Configured classifiers:', ', '.join(MODELS))

## 4. First pass — **all 17 original predictors, all four models**

**Professor's instruction:** try all predictors and all required methods initially, even if model assumptions may be unsuitable. We record a failure instead of quietly changing model parameters. Model-specific adjustments come *after* this stage.

The all-predictor results are **diagnostic**, not yet evidence of a deployable model: the brief warns that `PageValues` may carry outcome-derived information.

In [ ]:
for name, estimator in MODELS.items():
    evaluate(f'ALL17 | {name}', estimator, ALL, stage='first pass: all 17')
display(pd.DataFrame(LEDGER).set_index('candidate').round(4))
for row in LEDGER:
    if row['status'] == 'FAILED':
        print('Fit failure recorded:', row['candidate'], row['detail'])

## 5. Feature availability and leakage review

The source brief explicitly treats `PageValues` as potentially outcome-derived Google Analytics information. A training-only comparison *with versus without* it tests how much the apparent ranking quality changes, but a large AUC difference **alone does not prove leakage**. The substantive reason for exclusion is that the measure may depend on transactions not available when support must be assigned.

To satisfy the first-pass requirement, we already attempted all 17 predictors above. For practical model development, the **16-predictor version excluding `PageValues`** is our planned operational specification.

In [ ]:
# Assess the inclusion/exclusion using the identical CV scheme on training only.
with_row = next(x for x in LEDGER if x['candidate'] == 'ALL17 | Logistic')
without_row = evaluate('Operational16 | Logistic raw', MODELS['Logistic'], SAFE, stage='post-leakage baseline')
if with_row['status'] == 'OK' and without_row['status'] == 'OK':
    print(f'Logistic CV AUC with PageValues: {with_row["CV AUC"]:.4f}')
    print(f'Logistic CV AUC without it:      {without_row["CV AUC"]:.4f}')
    print(f'AUC difference:                  {with_row["CV AUC"]-without_row["CV AUC"]:+.4f}')
print('Training association with outcome:')
display(pd.crosstab(Xtr['PageValues'].gt(0), ytr, normalize='columns').rename(
    index={False:'PageValues=0', True:'PageValues>0'}, columns={0:'Non-buyers', 1:'Buyers'}).round(3))
print('Decision: exclude PageValues from subsequent operational comparisons; reason is prediction-time availability, not AUC alone.')

## 6. Comparable operational baselines (16 predictors)

**Method:** all classifier families are compared on the same training folds. Confusion matrices and ROC curves use **one** set of out-of-fold (OOF) predictions per session. AUC fold averages and pooled OOF AUC need not be numerically identical.

Default QDA can have singular covariance after dummy encoding. If it fails or produces non-finite predictions, record that and then show a separately labelled **regularised QDA**; do not call the latter the default.

In [ ]:
BASELINE_LABELS = {}
for name, estimator in MODELS.items():
    label = f'Operational16 | {name} raw'
    row = without_row if name == 'Logistic' else evaluate(label, estimator, SAFE, stage='operational baseline')
    if row['status'] == 'OK':
        try:
            oof(label)
            BASELINE_LABELS[name] = label
        except Exception as e:
            row['status'] = 'FAILED_PREDICTION'; row['detail'] = str(e)[:120]
            print('Prediction failure:', name, row['detail'])
if 'QDA' not in BASELINE_LABELS:
    label = 'Operational16 | QDA regularised 0.05'
    row = evaluate(label, QuadraticDiscriminantAnalysis(reg_param=.05), SAFE,
                   stage='model-specific repair')
    if row['status'] == 'OK':
        try: oof(label); BASELINE_LABELS['QDA (regularised)'] = label
        except Exception as e: print('QDA regularisation did not resolve prediction:', str(e)[:120])
base_rows=[]
for name, label in BASELINE_LABELS.items():
    p = oof(label)
    row = next(x for x in LEDGER if x['candidate'] == label)
    base_rows.append({'model':name, 'mean fold AUC':row['CV AUC'], 'OOF AUC':roc_auc_score(ytr,p),
                      'Brier score':brier_score_loss(ytr,p), **confusion_rates(ytr,p,.5)})
BASELINES = pd.DataFrame(base_rows).set_index('model')
display(BASELINES.round(4))
fig, ax = plt.subplots(figsize=(7.2,5.1))
for name, label in BASELINE_LABELS.items():
    p = oof(label)
    fpr, tpr, _ = roc_curve(ytr,p)
    ax.plot(fpr,tpr,label=f'{name} (AUC={roc_auc_score(ytr,p):.3f})')
    r=confusion_rates(ytr,p,.5)
    ax.plot(1-r['specificity'],r['sensitivity'],'o')
ax.plot([0,1],[0,1],'k--',lw=.8)
ax.set(xlabel='1 - specificity',ylabel='Sensitivity',title='Operational baselines: OOF ROC (dots: threshold 0.5)')
ax.legend(fontsize=8);plt.show()

## 7. Measured model weaknesses and implications

The professor's first-pass instruction is satisfied; **now** we can diagnose reasons for poor fit.

- **Gaussian NB:** class-conditional independence plus a Gaussian distribution for each numerical predictor.
- **LDA:** class-conditional multivariate normality with a shared covariance matrix.
- **QDA:** class-conditional multivariate normality, potentially different covariance matrices; more parameters.
- **Logistic:** a linear relationship between predictors and the *log-odds*, not necessarily a Gaussian distribution for predictors.

Diagnostics guide candidate changes but **do not** establish that a transformation will improve AUC; CV must answer that.

In [ ]:
assumption_rows=[]
for col in NUM_SAFE:
    for outcome in (0,1):
        arr=Xtr.loc[ytr.eq(outcome),col].to_numpy()
        assumption_rows.append({'variable':col,'class':outcome,'skew':stats.skew(arr),
                                'zero %':100*np.mean(arr==0)})
ASSUMPTIONS=pd.DataFrame(assumption_rows)
display(ASSUMPTIONS.pivot(index='variable',columns='class',values=['skew','zero %']).round(2))
for outcome in (0,1):
    corr=Xtr.loc[ytr.eq(outcome),NUM_SAFE].corr()
    top=corr.abs().where(~np.eye(len(corr),dtype=bool)).stack().sort_values(ascending=False)
    print(f'Class {outcome}: strongest absolute numeric within-class correlation: {top.index[0]} = {top.iloc[0]:.3f}')
print('Interpretation: skewness motivates log/power transforms; correlation warns Gaussian NB independence may be imperfect.')

## 8. CV-guided improvements: transformations, nonlinearity and tuning

**Design:** compare small, justified alternatives on identical stratified folds, retain every attempted specification in a results ledger. Transformation parameters are fitted inside the Pipeline, not across the full dataset.

- Skewness/zeros → `log1p`, Yeo–Johnson and a fold-fitted zero indicator; quantile mapping is included because the brief listed it.
- Nonlinear relationships → numerical second-order interactions (not an entirely new classifier).
- Tuning → regularisation of logistic regression, LDA shrinkage, Gaussian NB smoothing and QDA regularisation.
- Parsimony → predeclared five-variable alternative (at least five original predictors), compared with the full operational 16. This is a CV comparison, **not proof those variables are causally important**.

This is a bounded project-motivated comparison, not an exhaustive search over all possible models.

In [ ]:
CANDIDATES = {name: label for name,label in BASELINE_LABELS.items()}
# Transformation comparisons for each of the three required families.
for transform in ['log1p','yeojohnson','quantile','zeroind']:
    for name in ['Logistic','Gaussian NB','LDA']:
        label=f'{name} | {transform}'
        row=evaluate(label, MODELS[name], SAFE, transform=transform, stage='transformation')
        if row['status']=='OK': CANDIDATES[label]=label
# Nonlinear candidate and an interpretable predeclared subset (all preprocessing within CV).
five=['ExitRates','ProductRelated_Duration','Month','TrafficType','VisitorType']
for label, model, columns, transform, poly in [
    ('Logistic | raw interactions', MODELS['Logistic'], SAFE, 'raw', True),
    ('Logistic | five original predictors', MODELS['Logistic'], five, 'raw', False),
    ('LDA | yeojohnson interactions', MODELS['LDA'], SAFE, 'yeojohnson', True),
]:
    row=evaluate(label,model,columns,transform,poly,stage='nonlinearity or subset')
    if row['status']=='OK': CANDIDATES[label]=label
for label, model, transform in [
    ('Logistic | L2 C=0.1',LogisticRegression(C=.1,max_iter=3000),'raw'),
    ('Logistic | L2 C=10',LogisticRegression(C=10,max_iter=3000),'raw'),
    ('Gaussian NB | smoother',GaussianNB(var_smoothing=1e-5),'raw'),
    ('LDA | shrinkage',LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto'),'raw'),
    ('QDA | regularised 0.05',QuadraticDiscriminantAnalysis(reg_param=.05),'raw'),
]:
    row=evaluate(label,model,SAFE,transform,stage='tuning')
    if row['status']=='OK':CANDIDATES[label]=label
LED=pd.DataFrame(LEDGER)
display(LED[['stage','candidate','CV AUC','CV SD','status','warnings','detail']].round(4))
print('Successful candidates:', int(LED['status'].eq('OK').sum()), ' / ', len(LED),'attempted')

## 9. Model comparison and selection on training CV

**Why this approach?** The brief prioritises ROC-AUC but also asks for a cost-based interpretation. We select the highest CV-AUC configuration from the *operational*, leakage-screened candidates (not the all-17 diagnostic fits). Cost is then assessed for **each family** using its OOF probabilities and a threshold chosen on training data only.

**Caution:** Comparing many specifications on the same folds induces selection optimism. We do not claim a separate unbiased CV estimate after searching; the historical holdout is also previously viewed. These limitations must be reported rather than hidden.

In [ ]:
valid=LED.loc[LED['candidate'].isin(CANDIDATES.values()) & LED.status.eq('OK')].copy()
valid=valid.sort_values(['CV AUC','candidate'],ascending=[False,True])
WINNER=valid.iloc[0]['candidate']
print('Selected by predeclared primary metric (mean CV AUC):',WINNER,
      f"AUC={valid.iloc[0]['CV AUC']:.4f}")
display(valid[['candidate','CV AUC','CV SD','stage']].head(12).round(4))
# Compare one representative per required family, including any successful transformations/tuning.
def family_of(label):
    return next((name for name in MODELS if label.startswith(name+' |') or
                 label == f'Operational16 | {name} raw'),None)
family_rows=[]
for family in MODELS:
    rows=valid.loc[valid.candidate.map(family_of).eq(family)]
    if rows.empty: continue
    r=rows.iloc[0]; name=r['candidate']
    try: p=oof(name)
    except Exception as e:
        print('Unable to produce out-of-fold probabilities for',name,repr(e));continue
    t=optimal_threshold(ytr,p)
    family_rows.append({'family':family,'candidate':name,'CV AUC':r['CV AUC'],
                        'threshold':t,'cost/session':cost_at(ytr,p,t),
                        'sensitivity':confusion_rates(ytr,p,t)['sensitivity'],
                        'specificity':confusion_rates(ytr,p,t)['specificity']})
FAMILY=pd.DataFrame(family_rows).set_index('family')
display(FAMILY.round(4))
print('AUC winner and cheapest candidate may differ; cost and ranking are separate grounds.')

## Extension 1 — Cost-justified operating threshold

**Goal:** select a threshold other than 0.5 by explicitly trading off errors. A *lower* threshold increases sensitivity at the cost of specificity. Threshold changes alone do **not** change ROC-AUC, because predicted probability ranks are unchanged.

Under the brief's simplified costs (`C_FN=5`, `C_FP=1`) and **calibrated** probabilities, the theoretical cost-minimising threshold is

\[
 t^*=rac{C_{FP}}{C_{FN}+C_{FP}}=rac{1}{1+5}pprox0.167.
\]

We also estimate a cost-minimising threshold empirically using only OOF training probabilities. We then stress-test other ratios **without treating those ratios as measured business costs**. The model chosen by AUC remains fixed at this stage.

In [ ]:
P_TRAIN=oof(WINNER)
T_OPT=optimal_threshold(ytr,P_TRAIN)
GRID=np.linspace(0.0,1.0,101)
SENS=[confusion_rates(ytr,P_TRAIN,t)['sensitivity'] for t in GRID]
SPEC=[confusion_rates(ytr,P_TRAIN,t)['specificity'] for t in GRID]
COST=[cost_at(ytr,P_TRAIN,t) for t in GRID]
fig,axes=plt.subplots(1,2,figsize=(12,3.8))
axes[0].plot(GRID,SENS,label='Sensitivity')
axes[0].plot(GRID,SPEC,label='Specificity')
axes[0].axvline(T_OPT,color='black',ls='--',label=f'empirical t={T_OPT:.2f}')
axes[0].set(title='Error rates by threshold',xlabel='Threshold',ylabel='Fraction')
axes[0].legend()
axes[1].plot(GRID,COST,label='Weighted error cost')
axes[1].axvline(T_OPT,color='black',ls='--',label=f'empirical t={T_OPT:.2f}')
axes[1].axvline(1/6,color='grey',ls=':',label='calibrated-theory t*=0.167')
axes[1].set(title='Expected cost by threshold (5:1 assumption)',xlabel='Threshold',ylabel='Cost per session')
axes[1].legend(fontsize=8)
plt.tight_layout();plt.show()
comparison=[]
for t in [.5,1/6,T_OPT]:
    comparison.append({'threshold':round(t,3),'cost/session':cost_at(ytr,P_TRAIN,t),
                       **confusion_rates(ytr,P_TRAIN,t)})
display(pd.DataFrame(comparison).round(4))
print(f'Cost reduction relative to threshold 0.5: {1-cost_at(ytr,P_TRAIN,T_OPT)/cost_at(ytr,P_TRAIN,.5):.2%}')
print('Compare against trivial actions: always negative cost=',round(cost_at(ytr,P_TRAIN,1.0),4),
      '; always positive cost=',round(cost_at(ytr,P_TRAIN,0.0),4))
stress=[]
for ratio in [1,2,3,5,8,10]:
    t=optimal_threshold(ytr,P_TRAIN,ratio)
    rr=confusion_rates(ytr,P_TRAIN,t)
    stress.append({'FN / FP cost ratio':ratio,'threshold':t,
                   'sensitivity':rr['sensitivity'],'specificity':rr['specificity'],
                   'cost/session':cost_at(ytr,P_TRAIN,t,ratio,1)})
display(pd.DataFrame(stress).round(4))
print(f'Chosen threshold fixed from training CV: {T_OPT:.2f}. This ratio is a scenario, not a real business-cost estimate.')

## Extension 2 — Cross-validation versus held-out results

**Procedure:** both the selected model and threshold are fixed *before* this section; refit the specified pipeline on the entire 75% training portion and **only then** evaluate the reserved 25% test partition. No parameter or threshold is retuned using the test results.

**Interpretation caution:** this exact test partition was already inspected in the old notebook. Consequently, the comparison remains useful as a reproducibility exercise but **must not be described as a first independent confirmatory test**. A future genuinely independent evaluation would require new data or a defensible additional test design.

In [ ]:
FINAL_PIPE=clone(CONFIGS[WINNER]).fit(Xtr,ytr)
P_TEST=FINAL_PIPE.predict_proba(Xte)[:,1]
TRAIN_R=confusion_rates(ytr,P_TRAIN,T_OPT)
TEST_R=confusion_rates(yte,P_TEST,T_OPT)
TEST_AUC=roc_auc_score(yte,P_TEST)
CV_AUC=float(valid.iloc[0]['CV AUC'])
AUDIT=pd.DataFrame([
    {'evaluation':'Training out-of-fold 5-fold CV','AUC':roc_auc_score(ytr,P_TRAIN),
     'sensitivity':TRAIN_R['sensitivity'],'specificity':TRAIN_R['specificity'],
     'cost/session':cost_at(ytr,P_TRAIN,T_OPT)},
    {'evaluation':'Historical held-out test','AUC':TEST_AUC,
     'sensitivity':TEST_R['sensitivity'],'specificity':TEST_R['specificity'],
     'cost/session':cost_at(yte,P_TEST,T_OPT)}
]).set_index('evaluation')
display(AUDIT.round(4))
print('Five-fold mean AUC used in selection:',round(CV_AUC,4))
print('AUC difference (OOF minus historical holdout):',round(AUDIT.iloc[0]['AUC']-AUDIT.iloc[1]['AUC'],4))
print('Chosen final model:',WINNER,' | pre-fixed threshold:',T_OPT)
print('Held-out confusion matrix (rows actual 0/1; columns predicted 0/1):')
display(pd.DataFrame([[TEST_R['TN'],TEST_R['FP']],[TEST_R['FN'],TEST_R['TP']]],
                     index=['Actual no purchase','Actual purchase'],
                     columns=['Predicted no','Predicted yes']))

## 10. Conclusions, reporting checklist and limitations

**What the project has now demonstrated (after running all cells):**

1. **EDA:** baseline class imbalance, skewness, zero-heavy measurements and differences between categories. These *suggest* changes; CV determines whether they help.
2. **Professor's all-predictor first pass:** all four families are attempted on the original 17 predictors, with failures retained.
3. **Practical baseline:** the suspected outcome-derived `PageValues` feature is excluded from an operational 16-predictor comparison.
4. **CV improvements:** justify, estimate and compare every proposed transform, interaction, tuning setting and small predictor subset; retain failures in the ledger.
5. **Model choice:** a transparent AUC ranking plus cost comparisons. Do not interpret a higher training/CV score as proof a model will improve future sales.
6. **Extension 1:** training-only threshold curve, sensitivity/specificity, cost trade-off, stress-testing of cost ratio.
7. **Extension 2:** historical test comparison with no *within-notebook* test-driven tuning and an explicit prior-inspection caveat.

**Business limitations:** the brief's false-negative-as-lost-sale narrative and 5:1 cost ratio are simplifying assumptions. Classification cannot establish that routing support causes a sale; there are no treatment or revenue counterfactuals. The group's later CAC/discount × AOV discussions require a separate, agreed decision/action definition before they can replace this scenario.

**Data limitations:** records are *sessions*; customer IDs are unavailable, so repeated sessions from one person cannot be ruled out or group-split. A session-level, single-retailer, historical dataset may not transfer to different sites or future periods. `PageValues` prediction-time availability requires operational validation.

**Statistical limitations:** performance estimates after trying multiple model variants may have selection optimism, and the test split was already examined in the preserved notebook. These are limitations to disclose, not grounds to adjust this same test repeatedly.

In [ ]:
print('PROJECT SUMMARY — computed from this notebook')
print('Selected model:', WINNER)
print(f'5-fold mean CV AUC: {CV_AUC:.4f}')
print(f'Threshold (C_FN/C_FP=5 assumption): {T_OPT:.3f}')
print(f'CV OOF sensitivity / specificity: {TRAIN_R["sensitivity"]:.3f} / {TRAIN_R["specificity"]:.3f}')
print(f'Historical holdout AUC: {TEST_AUC:.4f}')
print(f'Historical holdout sensitivity / specificity: {TEST_R["sensitivity"]:.3f} / {TEST_R["specificity"]:.3f}')
print('Experiment ledger entries:', len(LEDGER))
print('Do not describe the historical holdout as untouched or newly unseen.')